In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "academy"
BRONZE_SCHEMA = "lab4_bronze"
SILVER_SCHEMA = "lab4_silver"
TABLE_NAME = "netflix_titles"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{TABLE_NAME}"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{TABLE_NAME}"

In [0]:
bronze_df = spark.read.table(BRONZE_TABLE)
display(bronze_df)
bronze_df.printSchema()

In [0]:
cleaned_df = (
    bronze_df
        .withColumn("show_id", F.trim("show_id"))
        .withColumn("type", F.initcap(F.trim("type")))
        .withColumn("title", F.trim("title"))
        .withColumn("director", F.trim("director"))
        .withColumn("country", F.trim("country"))
        .withColumn("rating", F.upper(F.trim("rating")))
        .withColumn("date_added",  F.try_to_date(F.trim("date_added"), F.lit("MMMM d, yyyy")))
        .withColumn("release_year", F.col("release_year").cast("int"))
)

In [0]:
silver_df = (
    cleaned_df
    .withColumn(
        "audience_category",
        F.when(F.col("rating").isin("TV-Y", "TV-Y7", "G"), "Children")
         .when(F.col("rating").isin("TV-G", "PG", "TV-PG"), "Family")
         .when(F.col("rating").isin("PG-13", "TV-14"), "Teen")
         .when(F.col("rating").isin("R", "NC-17", "TV-MA"), "Adults")
         .otherwise("Unknown")
    )
    .withColumn(
        "release_period",
        F.when(F.col("release_year") < 2000, "Before 2000")
         .when(F.col("release_year") < 2010, "2000-2009")
         .when(F.col("release_year") < 2020, "2010-2019")
         .otherwise("2020+")
    )
    .withColumn("has_director", F.col("director").isNotNull())
    .withColumn("silver_created_at", F.current_timestamp())
    .withColumn("silver_updated_at", F.current_timestamp())
)

In [0]:
display(silver_df)

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)